In [13]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [20]:
# Подготовка данных (пример)
X = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\X_train_road.csv')
S = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\X_train_speed.csv')
y = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\y_train_wheel.csv')

In [21]:
X_train = torch.tensor(X.values, dtype=torch.float32).to(device='cuda')
S_train = torch.tensor(S.values, dtype=torch.float32).to(device='cuda')
y_tensor = torch.tensor(y.values, dtype=torch.float32).to(device='cuda')

In [24]:
print(len(X_train))
print(len(S_train))
print(len(y_tensor))

13038
13038
13038


In [26]:
X_tensor = torch.cat((X_train, S_train), dim=1)
print(X_tensor[0])
print(X_tensor[0].size())

tensor([ 0.,  0.,  0.,  ...,  0.,  0., 59.], device='cuda:0')
torch.Size([12289])


In [25]:
print(y_tensor[0])
print(y_tensor[0].size())

tensor([0.0053], device='cuda:0')
torch.Size([1])


In [27]:
dataset = TensorDataset(X_tensor, y_tensor)
train_loader = DataLoader(dataset, batch_size=64, shuffle=False)

In [34]:
class IS_LSTM_Net_v3(nn.Module):
    def __init__(self, input_size=12289, hidden_size_0=1024, hidden_size_1=512, hidden_size_2=128, num_layers=2):
        super(IS_LSTM_Net_v3, self).__init__()
        self.hidden_size_0 = hidden_size_0
        self.hidden_size_1 = hidden_size_1
        self.hidden_size_2 = hidden_size_2
        self.num_layers = num_layers
        # LSTM слой
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size_0,
                            num_layers=num_layers, batch_first=True)

        # Первый полносвязный слой
        self.fc1 = nn.Linear(hidden_size_0, hidden_size_1)
        # Второй полносвязный слой
        self.fc2 = nn.Linear(hidden_size_1, hidden_size_2)
        # Третий (финальный) полносвязный слой для получения одного выходного значения
        self.fc3 = nn.Linear(hidden_size_2, 1)

        # Функции активации для каждого слоя
        self.relu = nn.ReLU()

    def forward(self, combined_input):
        
        # Пропускаем через LSTM
        lstm_out, _ = self.lstm(combined_input)
        # Выбираем выход последнего временного шага
        lstm_out = lstm_out[:, -1]
        # Пропускаем через полносвязные слои с применением ReLU после каждого (кроме последнего)
        out = self.relu(self.fc1(lstm_out))
        out = self.relu(self.fc2(out))
        # Финальный слой не имеет ReLU после себя, применяем tanh для получения выхода в диапазоне [-1, 1]
        output = torch.tanh(self.fc3(out))
        return output

In [37]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = IS_LSTM_Net_v3().to(device)
criterion = nn.MSELoss()  # Используйте подходящую функцию потерь
optimizer = optim.Adam(model.parameters(), lr=0.00001)

num_epochs = 1
for epoch in range(num_epochs):
    for inputs, labels in train_loader:
        inputs, labels = inputs.unsqueeze(1).to(device), labels.to(device)

        # Обнуление градиентов
        optimizer.zero_grad()

        # Прямой проход
        outputs = model(inputs)

        # Вычисление потерь
        loss = criterion(outputs, labels)

        # Обратное распространение и оптимизация
        loss.backward()
        optimizer.step()
    print(f'Epoch {epoch + 1}, Loss: {loss.item()}')


Epoch 1, Loss: 3.4045111533487216e-05


In [38]:
torch.save(model.state_dict(), 'C:\PycharmProjects\ETS_Autopilot\static\weight_model\weight_wheel_nn_lstm_3.pth')